# Day 05 — Local vector search and the data lifecycle: an index you can filter, save, update and delete from

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-05/LEARN.md`](../docs/day-05/LEARN.md).*

Day 04 turned Day 03's 44 chunks into 44 vectors and searched them by scoring every one. That is already a vector search. What it lacked is everything around the search that a real collection of documents needs: a way to **narrow** results by metadata, to **save** the index and load it safely, and to keep it **in step with the documents** as they are edited, retired and re-permissioned.

Today builds that, in plain NumPy, and then breaks it on purpose:

```
 Day 03 chunks ──> Day 04 model ──> VectorIndex: vectors + records + model  ──> search(question, filter)
                                         │  save / load (one file, atomic swap)
 documents change ──> re-ingest ──> sync: add · re-embed · update metadata · delete
```

| This notebook **does** | This notebook **does not** |
|---|---|
| Build an exact vector index from scratch: one vector and one record per chunk, keyed by `chunk_id` | Use a vector database or an approximate index (section 7 measures when you'd need one) |
| Filter by metadata (document, review date, permissions) before or after the ranking, and show the difference | Generate answers with an LLM (Day 06) |
| Save the index as one file with an atomic swap, and refuse to load it with the wrong model | Evaluate retrieval quality (Day 07) |
| Keep the index in step with changing documents: re-embed only changed text, update permissions without re-embedding, delete what's gone | Implement production access control: identity, tenants, audit (Day 14) |
| Show four ways a permission filter fails as access control, and the shape of a check that holds | Call a paid API or send your text to anyone |

**Cost:** free, no new packages. Uses Day 04's model (133 MB, downloaded once if it isn't in `models/` yet). **Runtime:** about a minute. **Requirements:** Python 3.11+ and `requirements.txt`.
**How to run:** open this notebook from inside the cloned repository, then *Kernel → Restart Kernel and Run All Cells*.
**Verified:** executed top to bottom on 2026-10-18 on Linux with Python 3.11, 3.12, 3.13 and 3.14, from both the repo root and `notebooks/`; the saved outputs come from the Python 3.13 run. In that environment the model files came from a copy matching the published SHA-256 values, because Hugging Face wasn't reachable; Day 04's download from Hugging Face has since run on Windows 11 and in CI on Linux, Windows and macOS. CI runs this notebook on all three on every push.

## 0. Setup

No new packages today. The shared code in [`src/rag_from_scratch/`](../src/rag_from_scratch/) grows by one module: **Day 04's embedding model** (`embeddings.py`), so this notebook doesn't repeat the download, hash check and `embed()` code. As with every shared module, a test checks that it returns exactly the vectors the Day 04 notebook computes.

In [1]:
import json
import os
import platform
import shutil
import statistics
import sys
import tempfile
import time
import warnings
from collections import Counter
from dataclasses import asdict, dataclass, fields, replace
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
except ImportError as error:
    raise ImportError("Day 05 uses Day 04's packages (numpy, onnxruntime, tokenizers, huggingface-hub). From the "
                      "repository root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.chunking import Chunk, section_chunks   # Day 03's chunker
from rag_from_scratch.embeddings import Embedder             # Day 04's model (new in src/ today)
from rag_from_scratch.ingest import ingest                   # Day 02's pipeline
from rag_from_scratch.probes import DAY03_PROBES             # Day 03's 13 questions

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__}")

Python 3.13.16 | numpy 2.4.6 | onnxruntime 1.31.0


In [2]:
TOP_K = 3                                     # chunks returned per question, as on Days 01, 03 and 04
OUT_DIR = REPO_ROOT / "outputs" / "day-05"    # outputs/ is git-ignored
INDEX_PATH = OUT_DIR / "index.npz"
REVIEWED_SINCE = "2025-10-13"                 # freshness filter: reviewed in the 365 days before Day 02's snapshot

## 1. The input: Day 03's chunks and Day 04's model

The same 44 chunks as Days 03 and 04, and the same model at the same pinned revision. `Embedder` checks both model files against their SHA-256 values before loading them, and downloads them only if they are missing.

In [3]:
documents, quarantined = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
print(f"{len(documents)} documents -> {len(chunks)} chunks\n")

embedder = Embedder(REPO_ROOT)     # first run downloads the model (133 MB); every run checks both SHA-256 values
started = time.perf_counter()
chunk_vectors = embedder.embed([c.text for c in chunks])
print(f"\nembedded {len(chunks)} chunks in {time.perf_counter() - started:.1f} s -> matrix {chunk_vectors.shape}, "
      f"{chunk_vectors.dtype}")

11 documents -> 44 chunks

tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified
onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified



embedded 44 chunks in 0.9 s -> matrix (44, 384), float32


## 2. A vector index is a matrix plus a list

Strip away the product names and a vector index stores three things:

- **vectors**: a matrix with one row per chunk (here 44 × 384 numbers);
- **records**: for each row, everything you need besides the vector: the `chunk_id` (the primary key), the text, and the metadata Day 02 and Day 03 attached (`allowed_groups`, `last_reviewed`, `source_path`, `doc_hash`, `chunk_hash`, ...);
- **the model** that produced the vectors, because a query vector from any other model is meaningless against them (Day 04).

The class below is that, plus four operations: insert-or-replace (`upsert`), `update_metadata`, `delete`, and `search`. `search` is exact: it scores every candidate, as Day 04 did. Vector databases add approximate indexes for speed at millions of rows (section 7), but the contract is the same.

In [4]:
@dataclass(frozen=True)
class Result:
    rank: int
    score: float     # cosine similarity (the vectors have length 1)
    id: str          # chunk_id
    chunk: Chunk     # the stored record: text, permissions, provenance


def top_k(scores: np.ndarray, k: int) -> np.ndarray:
    # Positions of the k highest scores, best first. argpartition finds them without sorting everything.
    best = np.argpartition(-scores, k)[:k] if k < len(scores) else np.arange(len(scores))
    return best[np.argsort(-scores[best], kind="stable")]


class VectorIndex:
    """A flat (exact) vector index: one vector and one record per chunk, plus the model that made the vectors."""

    def __init__(self, model: str, revision: str, dimensions: int):
        self.model, self.revision, self.dimensions = model, revision, dimensions
        self.vectors = np.zeros((0, dimensions), dtype=np.float32)   # row i is the vector of records[i]
        self.records: list[Chunk] = []
        self.row: dict[str, int] = {}                                # chunk_id -> row: the primary key

    def __len__(self) -> int:
        return len(self.records)

    def upsert(self, chunks: list[Chunk], vectors: np.ndarray) -> None:
        # Insert chunks with a new chunk_id; replace the record and the vector of an existing one.
        if len(chunks) != len(vectors) or vectors.shape[1] != self.dimensions:
            raise ValueError(f"{len(chunks)} chunks, vectors of shape {vectors.shape}")
        new_vectors = []
        for chunk, vector in zip(chunks, vectors):
            if chunk.chunk_id in self.row:
                i = self.row[chunk.chunk_id]
                self.records[i], self.vectors[i] = chunk, vector
            else:
                self.row[chunk.chunk_id] = len(self.records)
                self.records.append(chunk)
                new_vectors.append(vector)
        if new_vectors:
            self.vectors = np.vstack([self.vectors, np.asarray(new_vectors, dtype=np.float32)])

    def update_metadata(self, chunk: Chunk) -> None:
        # Same text, new permissions or provenance: replace the record, keep the vector.
        i = self.row[chunk.chunk_id]
        if self.records[i].chunk_hash != chunk.chunk_hash:
            raise ValueError(f"{chunk.chunk_id}: the text changed, so the vector must be recomputed")
        self.records[i] = chunk

    def delete(self, chunk_ids) -> int:
        # Remove rows and close the gaps. (Large stores mark rows as deleted and compact later.)
        gone = {c for c in chunk_ids if c in self.row}
        keep = [i for i, r in enumerate(self.records) if r.chunk_id not in gone]
        self.vectors = self.vectors[keep]
        self.records = [self.records[i] for i in keep]
        self.row = {r.chunk_id: i for i, r in enumerate(self.records)}
        return len(gone)

    def search(self, query_vector: np.ndarray, k: int = TOP_K, where: dict | None = None,
               filter_mode: str = "pre") -> list[Result]:
        # Exact search: score every candidate row, keep the best k. A filter (section 3) runs before
        # the ranking ("pre") or on the ranked top k ("post").
        if filter_mode == "pre":
            rows = np.array([i for i, r in enumerate(self.records) if where is None or matches(r, where)],
                            dtype=np.int64)
            scores = self.vectors[rows] @ query_vector
            picked = [(rows[j], scores[j]) for j in top_k(scores, k)]
        elif filter_mode == "post":
            scores = self.vectors @ query_vector
            picked = [(i, scores[i]) for i in top_k(scores, k) if where is None or matches(self.records[i], where)]
        else:
            raise ValueError(f"filter_mode must be 'pre' or 'post', not {filter_mode!r}")
        return [Result(rank, float(score), self.records[i].chunk_id, self.records[i])
                for rank, (i, score) in enumerate(picked, start=1)]

In [5]:
index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
index.upsert(chunks, chunk_vectors)
print(f"{len(index)} rows | vectors {index.vectors.shape} {index.vectors.dtype}, {index.vectors.nbytes:,} bytes "
      f"| model {index.model} @ {index.revision[:12]}\n")

record = index.records[index.row["manager-pay-review-guide#overview"]]
for field, value in asdict(record).items():
    if field == "text":
        value = value.splitlines()[0] + " / ..."
    elif field.endswith("_hash"):
        value = value[:16] + "..."
    print(f"  {field:<15}{value}")

44 rows | vectors (44, 384) float32, 67,584 bytes | model BAAI/bge-small-en-v1.5 @ 5c38ec7c405e

  chunk_id       manager-pay-review-guide#overview
  doc_id         manager-pay-review-guide
  section        Overview
  text           Manager guide: annual pay review > Overview / ...
  title          Manager guide: annual pay review
  allowed_groups ('managers', 'people-team')
  source_path    data/sample/acme-handbook/manager-pay-review-guide.txt
  last_reviewed  None
  doc_hash       253f87cd0ab1383c...
  chunk_hash     146c27100b941450...


A record carries everything a later step needs: what to show and cite (`text`, `chunk_id`, `title`), how fresh it is (`last_reviewed`), who may see it (`allowed_groups`), and which version of the text produced the vector (`chunk_hash`). This one belongs to the only managers-only document in the corpus; we'll come back to it.

Searching the index must give exactly Day 04's results, since it is the same maths in a new container:

In [6]:
def show(results: list[Result], live: set[str] | None = None) -> None:
    # One line per result; flags restricted chunks and, with `live`, chunks no document contains any more.
    if not results:
        print("    (no results)")
    for r in results:
        note = "" if "all-employees" in r.chunk.allowed_groups else f"   <- only {', '.join(r.chunk.allowed_groups)}"
        if live is not None and r.id not in live:
            note += "   <- no longer in any document"
        print(f"    #{r.rank} {r.score:.2f}  [{r.id}]{note}")


QUESTION = "How much time off do new dads get?"
print(QUESTION)
show(index.search(embedder.embed_query(QUESTION)))

# The index must return what Day 04's brute-force search returned: the same chunks in the same order
for probe in DAY03_PROBES:
    q = embedder.embed_query(probe.question)
    brute_force = [chunks[i].chunk_id for i in np.argsort(-(chunk_vectors @ q), kind="stable")[:TOP_K]]
    assert [r.id for r in index.search(q)] == brute_force, probe.question
print(f"\nsame top {TOP_K} as Day 04's brute-force search for all {len(DAY03_PROBES)} of Day 03's questions")

How much time off do new dads get?
    #1 0.72  [parental-leave-policy#returning-to-work]
    #2 0.67  [parental-leave-policy#who-is-eligible]
    #3 0.66  [parental-leave-policy#leave-entitlement]

same top 3 as Day 04's brute-force search for all 13 of Day 03's questions


Same three parental-leave chunks, same scores as Day 04, and the same top 3 as a brute-force search for every Day 03 question. One implementation detail: `top_k` uses `np.argpartition`, which finds the best k without sorting all the scores. That matters at 100,000 rows, not at 44.

## 3. Metadata filters

A **filter** restricts the search to records whose metadata satisfies a condition. Three common uses, all possible because ingestion kept the metadata on every chunk:

- **scope**: search one document or one department (`doc_id`);
- **freshness**: only policies reviewed recently (`last_reviewed`);
- **permissions**: only chunks the user's groups may read (`allowed_groups`), the "security trimming" pattern.

Vector databases express filters as small declarative conditions that run inside the database. `matches()` is a tiny version with three kinds of condition: equality, "any of" for lists, and "at least".

In [7]:
def matches(chunk: Chunk, where: dict) -> bool:
    # where = {field: condition}; every condition must hold. A condition is
    #   a value         -> equal to it (for a list field such as allowed_groups: contained in it)
    #   {"any": [...]}  -> shares at least one value with the list
    #   {"gte": value}  -> at least this value; a missing value (None) fails
    for field, condition in where.items():
        value = getattr(chunk, field)              # a misspelled field raises instead of matching nothing
        values = set(value) if isinstance(value, tuple) else {value}
        if isinstance(condition, dict):
            if set(condition) - {"any", "gte"}:
                raise ValueError(f"unknown operator in {condition}")
            if "any" in condition and not values & set(condition["any"]):
                return False
            if "gte" in condition and (value is None or value < condition["gte"]):
                return False
        elif condition not in values:
            return False
    return True


FILTERS = {
    "scope: one document": {"doc_id": "remote-work-policy"},
    "freshness: reviewed in the last year": {"last_reviewed": {"gte": REVIEWED_SINCE}},
    "permissions: an employee": {"allowed_groups": {"any": ["all-employees"]}},
    "permissions: a manager": {"allowed_groups": {"any": ["all-employees", "managers"]}},
}
for name, where in FILTERS.items():
    passing = sum(matches(r, where) for r in index.records)
    print(f"{name:<38}{passing:>3} of {len(index)} chunks pass")

fresh = FILTERS["freshness: reviewed in the last year"]
hidden = sorted({(r.doc_id, r.last_reviewed) for r in index.records if not matches(r, fresh)}, key=str)
print(f"\nthe freshness filter hides: {', '.join(f'{d} (last reviewed: {when})' for d, when in hidden)}")

ADMIN_QUESTION = "Who approves admin rights?"
q = embedder.embed_query(ADMIN_QUESTION)
print(f"\n{ADMIN_QUESTION}\n  no filter")
show(index.search(q))
print("  freshness filter")
show(index.search(q, where=fresh))

scope: one document                     5 of 44 chunks pass
freshness: reviewed in the last year   39 of 44 chunks pass
permissions: an employee               43 of 44 chunks pass
permissions: a manager                 44 of 44 chunks pass

the freshness filter hides: it-software-requests (last reviewed: 2025-06-30), manager-pay-review-guide (last reviewed: None)

Who approves admin rights?
  no filter
    #1 0.67  [it-software-requests#approval-times]
    #2 0.60  [expense-reimbursement-policy#approval-and-payment]
    #3 0.56  [manager-pay-review-guide#overview]   <- only managers, people-team
  freshness filter
    #1 0.60  [expense-reimbursement-policy#approval-and-payment]
    #2 0.54  [new-starter-it-checklist#accounts-and-access]
    #3 0.54  [expense-reimbursement-policy#scope]


A filter is a **hard rule**, not a preference. The freshness filter hides two documents (5 of the 44 chunks): the software-requests page, last reviewed on 2025-06-30 and flagged as overdue on Day 02, and the manager guide, which has no review date. So for *"Who approves admin rights?"* the only chunk that answers (the software-requests table) can no longer be returned, and the search fills the top 3 with chunks that don't answer. That may be the right policy (don't answer from an overdue document) or a silent loss of a correct answer. Decide it deliberately, and show users why something is missing.

Note the manager guide: it has no review date at all, and `{"gte": ...}` treats a missing value as failing. Missing metadata decides what is retrievable, which is why Day 02 warned about it.

### 3.1 Before or after the ranking?

There are two places to apply a filter:

- **pre-filter**: keep only the matching records, then rank them and take the top k;
- **post-filter**: rank everything, take the top k, then drop those that don't match.

Post-filtering is tempting because it is cheaper in approximate indexes (rank once, filter a few results). Watch what it does here:

In [8]:
CASES = [("When do pay rises take effect?", "permissions: an employee"),
         ("How do I connect to the VPN?", "scope: one document")]
for question, filter_name in CASES:
    q = embedder.embed_query(question)
    print(f"{question}   [filter: {filter_name}]\n  no filter")
    show(index.search(q))
    for mode in ("post", "pre"):
        results = index.search(q, where=FILTERS[filter_name], filter_mode=mode)
        print(f"  {mode}-filter: {len(results)} of {TOP_K} results")
        show(results)
    print()

When do pay rises take effect?   [filter: permissions: an employee]
  no filter
    #1 0.70  [manager-pay-review-guide#overview]   <- only managers, people-team
    #2 0.59  [expense-reimbursement-policy#approval-and-payment]
    #3 0.58  [parental-leave-policy#returning-to-work]
  post-filter: 2 of 3 results
    #1 0.59  [expense-reimbursement-policy#approval-and-payment]
    #2 0.58  [parental-leave-policy#returning-to-work]
  pre-filter: 3 of 3 results
    #1 0.59  [expense-reimbursement-policy#approval-and-payment]
    #2 0.58  [parental-leave-policy#returning-to-work]
    #3 0.57  [expense-reimbursement-policy#submitting-a-claim]

How do I connect to the VPN?   [filter: scope: one document]
  no filter
    #1 0.80  [vpn-access-guide#troubleshooting]
    #2 0.76  [vpn-access-guide#connecting]
    #3 0.74  [vpn-access-guide#when-to-use-the-vpn]
  post-filter: 0 of 3 results
    (no results)
  pre-filter: 3 of 3 results
    #1 0.71  [remote-work-policy#security-while-remote]
    #2 0

- **Permissions, post-filter:** the managers-only chunk is the best match for the pay question, so it takes one of the top 3 places and is then removed: the employee gets **2 results instead of 3**. Note also what the unfiltered list shows: without a filter, the restricted chunk is the first result.
- **Scope, post-filter:** the top 3 for the VPN question all come from the VPN guide, so a search scoped to the remote work policy returns **nothing**, although that policy has a chunk about working securely with the VPN. The pre-filter finds it at #1.

Post-filtering returns *fewer than k results, possibly zero,* whenever the filter is selective. Asking for more than k and filtering afterwards ("over-fetching") reduces the problem but can't remove it. With an exact index, pre-filtering costs nothing extra: it scores fewer rows. With an approximate index, pre-filtering is harder, which is why vector databases document their filter modes carefully (see LEARN.md, section 5).

## 4. Saving and loading the index

An index that lives only in memory is rebuilt on every start, which means re-embedding every chunk. So we save it, and three things matter:

1. **One file, swapped in atomically.** Write the new index to a temporary file in the same folder, then `os.replace()` it over the old one. Python documents that rename as atomic on POSIX systems (Linux, macOS), so a reader sees the old index or the new one, never half of each. On Windows it also replaces the file in one call, without that written guarantee. The vectors and the records live in the same file, so they can't get out of step.
2. **No pickle.** The file is a `.npz` (a zip of NumPy arrays): the vectors as `float32`, the records and a manifest as UTF-8 JSON bytes. `np.load(..., allow_pickle=False)` refuses anything that could run code when loaded.
3. **The model travels with the vectors.** The manifest records the model and revision, and `load_index()` refuses to pair the vectors with queries from another model.

In [9]:
INDEX_FORMAT = 1   # change it when the file layout changes, so an old file is rejected instead of misread


def write_index(f, index: VectorIndex) -> None:
    # One .npz file (a zip of NumPy arrays): the vectors, plus the manifest and the records as UTF-8 JSON bytes.
    manifest = {"format": INDEX_FORMAT, "model": index.model, "revision": index.revision,
                "dimensions": index.dimensions, "normalised": True,
                "similarity": "cosine (dot product of unit vectors)", "count": len(index)}
    records = [asdict(r) for r in index.records]
    as_bytes = lambda obj: np.frombuffer(json.dumps(obj, ensure_ascii=False).encode("utf-8"), dtype=np.uint8)
    np.savez(f, vectors=index.vectors, manifest=as_bytes(manifest), records=as_bytes(records))


def save_index(index: VectorIndex, path: Path) -> None:
    # Write a complete new file next to the old one, then swap it in with one os.replace call:
    # anyone loading the index gets the old version or the new one, never a mix or half a file.
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")      # same folder, so the rename stays on one file system
    try:
        with tmp.open("wb") as f:
            write_index(f, index)
            f.flush()
            os.fsync(f.fileno())                   # on the disk before the swap, not just in a buffer
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)                # still there only if something failed


def load_index(path: Path, model: str, revision: str) -> VectorIndex:
    with np.load(path, allow_pickle=False) as data:      # allow_pickle=False: a data file must never run code
        manifest = json.loads(data["manifest"].tobytes().decode("utf-8"))
        records = json.loads(data["records"].tobytes().decode("utf-8"))
        vectors = data["vectors"]
    if manifest["format"] != INDEX_FORMAT:
        raise ValueError(f"{path.name}: index format {manifest['format']}, this code reads format {INDEX_FORMAT}")
    if (manifest["model"], manifest["revision"]) != (model, revision):
        raise ValueError(f"{path.name} was built with {manifest['model']} @ {manifest['revision'][:12]}, but the "
                         f"queries come from {model} @ {revision[:12]}. Vectors from different models can't be "
                         "compared: rebuild the index.")
    if vectors.shape != (manifest["count"], manifest["dimensions"]) or len(records) != manifest["count"]:
        raise ValueError(f"{path.name}: {len(records)} records and vectors {vectors.shape}, but the manifest "
                         f"says {manifest['count']}")
    index = VectorIndex(model, revision, manifest["dimensions"])
    index.upsert([Chunk(**{**r, "allowed_groups": tuple(r["allowed_groups"])}) for r in records], vectors)
    if len(index) != manifest["count"]:
        raise ValueError(f"{path.name}: duplicate chunk IDs")
    return index


save_index(index, INDEX_PATH)
print(f"saved {len(index)} chunks to {INDEX_PATH.relative_to(REPO_ROOT).as_posix()} "
      f"({INDEX_PATH.stat().st_size:,} bytes):")
with np.load(INDEX_PATH, allow_pickle=False) as data:
    for name in data.files:
        print(f"  {name:<9}{str(data[name].shape):<12}{data[name].dtype}")

reloaded = load_index(INDEX_PATH, embedder.model["repo"], embedder.revision)
same = reloaded.records == index.records and np.array_equal(reloaded.vectors, index.vectors)
print(f"\nreloaded: {len(reloaded)} chunks; records and vectors identical to the index in memory: {same}")
print("top 3 after reloading:", [r.id for r in reloaded.search(embedder.embed_query(QUESTION))])

saved 44 chunks to outputs/day-05/index.npz (99,192 bytes):
  vectors  (44, 384)   float32
  manifest (210,)      uint8
  records  (30636,)    uint8

reloaded: 44 chunks; records and vectors identical to the index in memory: True
top 3 after reloading: ['parental-leave-policy#returning-to-work', 'parental-leave-policy#who-is-eligible', 'parental-leave-policy#leave-entitlement']


The whole index is one file of 99,192 bytes: 67,584 bytes of vectors (44 × 384 × 4) and about 31 KB of records. Reloading it gives identical records and vectors, and the same results. Day 06 can load this file instead of re-embedding the chunks.

Why the temporary file? Compare what's left on disk when a save fails halfway: a naive save that writes straight into `index.npz`, and the atomic one. The bug is realistic: one record's groups stored as a Python `set`, which JSON can't encode.

In [10]:
def naive_save(index: VectorIndex, path: Path) -> None:
    with path.open("wb") as f:     # opening a file for writing empties it at once
        write_index(f, index)


# A realistic bug: one record's groups stored as a Python set, which JSON can't encode
broken = VectorIndex(index.model, index.revision, index.dimensions)
broken.upsert([replace(c, allowed_groups=set(c.allowed_groups)) if i == 0 else c for i, c in enumerate(chunks)],
              chunk_vectors)

crash_outcome = {}
for name, save in [("naive save", naive_save), ("atomic save", save_index)]:
    target = OUT_DIR / f"crash-demo-{name.split()[0]}.npz"
    save_index(index, target)                 # a good index is on disk...
    try:
        save(broken, target)                  # ...then a save fails halfway
    except TypeError as error:
        print(f"{name}: the save failed ({error})")
    try:
        crash_outcome[name] = len(load_index(target, index.model, index.revision))
        print(f"  the file on disk still loads: {crash_outcome[name]} chunks\n")
    except Exception as error:
        crash_outcome[name] = type(error).__name__
        print(f"  the file on disk is broken: {type(error).__name__}: {error}\n")
    target.unlink()
print("files left in outputs/day-05/:", sorted(p.name for p in OUT_DIR.iterdir()))

naive save: the save failed (Object of type set is not JSON serializable)
  the file on disk is broken: EOFError: No data left in file

atomic save: the save failed (Object of type set is not JSON serializable)
  the file on disk still loads: 44 chunks

files left in outputs/day-05/: ['index.npz']


Opening a file for writing empties it immediately, so when the naive save fails, the good index is already gone, and the next start fails too. The atomic save fails in the temporary file, which is cleaned up; the previous index is untouched. Production stores get the same guarantee from write-ahead logs or transactions; the idea is identical.

And the model check:

In [11]:
UPGRADED = "0123456789abcdef0123456789abcdef01234567"   # a made-up newer revision of the model
try:
    load_index(INDEX_PATH, embedder.model["repo"], UPGRADED)
except ValueError as error:
    refusal = str(error)
    print(f"refused: {refusal}")

refused: index.npz was built with BAAI/bge-small-en-v1.5 @ 5c38ec7c405e, but the queries come from BAAI/bge-small-en-v1.5 @ 0123456789ab. Vectors from different models can't be compared: rebuild the index.


Without that check, the index would load, every search would return three results with plausible-looking scores, and the ranking couldn't be trusted. Nothing crashes when vectors from two models are compared; this check is the only thing that notices.

## 5. Updates and deletes: keeping the index in step with the documents

Documents change after they are indexed: edited, retired, renamed, re-permissioned. Day 02 detected those changes per document; Day 03 gave every chunk a stable `chunk_id` and a `chunk_hash` of its text; Day 04 showed that a vector only has to be recomputed when the text changes. A **sync** puts this together. For every chunk the documents produce today, compared with the index by `chunk_id`:

| Situation | Action | Runs the model? |
|---|---|---|
| `chunk_id` not in the index | **add** | yes |
| same `chunk_id`, different `chunk_hash` | **re-embed** and replace | yes |
| same text, different metadata (permissions, dates, document version) | **update metadata**, keep the vector | no |
| identical | nothing | no |
| in the index, but in no document any more | **delete** | no |

`sync()` takes the **complete** list of current chunks. That makes deletes possible (anything missing is deleted), and dangerous: if ingestion fails and returns nothing, a naive sync deletes the whole index. So `sync()` refuses to delete more than half the index unless told otherwise.

In [12]:
def plan_sync(index: VectorIndex, current: list[Chunk]) -> dict[str, list[str]]:
    # Compare the chunks the documents produce now with what the index holds, chunk_id by chunk_id.
    by_id = {c.chunk_id: c for c in current}
    if len(by_id) != len(current):
        raise ValueError("duplicate chunk IDs in the new chunks")
    plan = {"add": [], "re-embed": [], "update metadata": [], "delete": [], "unchanged": []}
    for chunk_id, chunk in by_id.items():
        old = index.records[index.row[chunk_id]] if chunk_id in index.row else None
        if old is None:
            plan["add"].append(chunk_id)               # a new section or a new document
        elif old.chunk_hash != chunk.chunk_hash:
            plan["re-embed"].append(chunk_id)          # the text changed, so the stored vector is wrong
        elif old != chunk:
            plan["update metadata"].append(chunk_id)   # same text: new permissions, title, dates or doc version
        else:
            plan["unchanged"].append(chunk_id)
    plan["delete"] = sorted(set(index.row) - set(by_id))   # in the index, but in no document any more
    return plan


def sync(index: VectorIndex, current: list[Chunk], embed, max_delete_fraction: float = 0.5) -> dict[str, list[str]]:
    # Make the index match `current`, the COMPLETE list of chunks the documents produce now.
    plan = plan_sync(index, current)
    if len(index) and len(plan["delete"]) > max_delete_fraction * len(index):
        raise RuntimeError(f"refusing to delete {len(plan['delete'])} of {len(index)} chunks. Is the new chunk list "
                           "incomplete (a failed or partial ingest)? Raise max_delete_fraction if this is intended.")
    by_id = {c.chunk_id: c for c in current}
    to_embed = [by_id[i] for i in plan["add"] + plan["re-embed"]]
    if to_embed:                                       # the only step that runs the model
        index.upsert(to_embed, embed([c.text for c in to_embed]))
    for chunk_id in plan["update metadata"]:
        index.update_metadata(by_id[chunk_id])
    index.delete(plan["delete"])
    return plan

Now apply Day 02's five changes to a temporary copy of the sample data (an edited policy, a retired one, a new one, a permission change, a file re-saved with Windows line endings) plus a sixth: a renamed heading. Then re-ingest, re-chunk and sync the saved index.

In [13]:
NEW_POLICY = """---
title: Annual Leave Policy
owner: People Team
last_reviewed: 2026-10-01
---

## Entitlement
Full-time employees receive 25 days of paid annual leave per calendar year, plus public holidays.
"""


def edit(path: Path, old: str, new: str) -> None:
    text = path.read_text(encoding="utf-8")
    assert old in text, (path.name, old)
    path.write_text(text.replace(old, new), encoding="utf-8", newline="\n")


def day02_changes(sample: Path) -> None:
    # Day 02's five changes (its section 6), plus one renamed heading
    handbook, acl_path = sample / "acme-handbook", sample / "access-control.json"
    edit(handbook / "remote-work-policy.md", "USD 500", "USD 600")                  # 1. an edited policy
    (handbook / "vpn-access-guide.md").unlink()                                     # 2. a retired policy
    (handbook / "annual-leave-policy.md").write_text(NEW_POLICY, encoding="utf-8", newline="\n")   # 3. a new one
    acl = json.loads(acl_path.read_text(encoding="utf-8"))
    acl["permissions"]["acme-handbook/annual-leave-policy.md"] = ["all-employees"]
    acl["permissions"]["acme-handbook/password-and-mfa-standard.md"] = ["all-employees", "contractors"]  # 4. access
    acl_path.write_text(json.dumps(acl, indent=2), encoding="utf-8", newline="\n")
    expense = handbook / "expense-reimbursement-policy.md"                          # 5. re-saved with Windows
    expense.write_bytes(expense.read_bytes().replace(b"\n", b"\r\n"))                #    line endings
    edit(handbook / "security-incident-reporting.md", "## What not to do", "## Mistakes to avoid")   # 6. a heading


def corpus_after(change) -> list[Chunk]:
    # Apply `change` to a temporary copy of data/sample, then ingest and chunk the copy. The originals are untouched.
    with tempfile.TemporaryDirectory() as tmp:
        sample = Path(tmp) / "data" / "sample"
        shutil.copytree(REPO_ROOT / "data" / "sample", sample)
        change(sample)
        docs, _ = ingest(Path(tmp))
    return [c for d in docs for c in section_chunks(d)]


def by_document(chunk_ids: list[str]) -> str:
    # "vpn-access-guide (4), annual-leave-policy#entitlement": short lists for printing
    counts = Counter(i.split("#")[0] for i in chunk_ids)
    return ", ".join(f"{doc} ({n})" if n > 1 else next(i for i in chunk_ids if i.startswith(doc + "#"))
                     for doc, n in counts.items())


embedded = []                     # how many texts each sync sends through the model


def counting_embed(texts: list[str]) -> np.ndarray:
    embedded.append(len(texts))
    return embedder.embed(texts)


new_chunks = corpus_after(day02_changes)
live_ids = {c.chunk_id for c in new_chunks}
synced = load_index(INDEX_PATH, index.model, index.revision)          # start from the saved index
old_records = {r.chunk_id: r for r in synced.records}
scenario_plan = sync(synced, new_chunks, counting_embed)
scenario_embedded = sum(embedded)

for action, ids in scenario_plan.items():
    detail = by_document(ids) if action != "unchanged" else ""
    print(f"{action:<16}{len(ids):>3}  {detail}".rstrip())
new_by_id = {c.chunk_id: c for c in new_chunks}
changed_fields = Counter(f.name for i in scenario_plan["update metadata"] for f in fields(Chunk)
                         if getattr(old_records[i], f.name) != getattr(new_by_id[i], f.name))
print(f"\nmetadata updates by field: {dict(changed_fields)}")
print(f"texts sent through the model: {scenario_embedded} (a rebuild would embed all {len(new_chunks)})")
print(f"index: {len(old_records)} chunks before, {len(synced)} after")

add               2  annual-leave-policy#entitlement, security-incident-reporting#mistakes-to-avoid
re-embed          1  remote-work-policy#equipment-and-home-office
update metadata  11  password-and-mfa-standard (4), remote-work-policy (4), security-incident-reporting (3)
delete            5  security-incident-reporting#what-not-to-do, vpn-access-guide (4)
unchanged        27

metadata updates by field: {'allowed_groups': 4, 'doc_hash': 7}
texts sent through the model: 3 (a rebuild would embed all 41)
index: 44 chunks before, 41 after


Six changes, five kinds of work:

- **add 2:** the new annual leave policy, and the renamed section. Renaming *What not to do* to *Mistakes to avoid* changes its `chunk_id` (the ID comes from the heading), so to the index it is a new chunk.
- **re-embed 1:** only the home-office section of the remote work policy changed (USD 500 → USD 600).
- **update metadata 11, without the model:** contractors were added to the password standard (`allowed_groups` on its 4 chunks), and 7 unchanged sections of the two edited documents now point to the new document version (`doc_hash`).
- **delete 5:** the retired VPN guide's 4 chunks, and the old *What not to do* section.
- **unchanged 27:** including the expense policy, whose file bytes changed (Windows line endings) but whose normalised text didn't.

Three texts went through the model instead of 41. The deletes are the step that is easiest to forget, so here is the same update without them:

In [14]:
# The same updates without the delete step: insert or replace by chunk_id, never remove anything
upsert_only = load_index(INDEX_PATH, index.model, index.revision)
naive_plan = plan_sync(upsert_only, new_chunks)
todo = [new_by_id[i] for i in naive_plan["add"] + naive_plan["re-embed"]]
upsert_only.upsert(todo, embedder.embed([c.text for c in todo]))
for chunk_id in naive_plan["update metadata"]:
    upsert_only.update_metadata(new_by_id[chunk_id])
print(f"upsert only: {len(upsert_only)} chunks | full sync: {len(synced)} chunks "
      f"| the documents: {len(new_chunks)} chunks\n")

lifecycle_hits = {}
for question in ("How do I connect to the VPN?", "What should I avoid doing after a security incident?"):
    q = embedder.embed_query(question)
    print(question)
    for name, idx in [("upsert only", upsert_only), ("full sync", synced)]:
        lifecycle_hits[question, name] = idx.search(q)
        print(f"  {name}")
        show(lifecycle_hits[question, name], live=live_ids)
    print()

upsert only: 46 chunks | full sync: 41 chunks | the documents: 41 chunks

How do I connect to the VPN?
  upsert only
    #1 0.80  [vpn-access-guide#troubleshooting]   <- no longer in any document
    #2 0.76  [vpn-access-guide#connecting]   <- no longer in any document
    #3 0.74  [vpn-access-guide#when-to-use-the-vpn]   <- no longer in any document
  full sync
    #1 0.71  [remote-work-policy#security-while-remote]
    #2 0.58  [new-starter-it-checklist#your-first-day]
    #3 0.57  [password-and-mfa-standard#multi-factor-authentication]



What should I avoid doing after a security incident?
  upsert only
    #1 0.80  [security-incident-reporting#mistakes-to-avoid]
    #2 0.79  [security-incident-reporting#what-not-to-do]   <- no longer in any document
    #3 0.69  [security-incident-reporting#what-counts-as-an-incident]
  full sync
    #1 0.80  [security-incident-reporting#mistakes-to-avoid]
    #2 0.69  [security-incident-reporting#what-counts-as-an-incident]
    #3 0.68  [security-incident-reporting#after-you-report]



An index that only inserts and replaces holds **46** chunks for documents that now produce **41**. The retired VPN guide still answers VPN questions, in all three top places: a user would be sent to a procedure the company withdrew. The renamed section appears twice, old and new, in two of the three places with nearly identical text. Neither shows up as an error; the results look normal. The fully synced index returns the remote work policy's VPN advice instead, and the renamed section once.

Is the synced index really what a rebuild would produce?

In [15]:
rebuilt = VectorIndex(index.model, index.revision, index.dimensions)
rebuilt.upsert(new_chunks, embedder.embed([c.text for c in new_chunks]))

same_ids = set(synced.row) == set(rebuilt.row)
same_records = all(synced.records[synced.row[i]] == rebuilt.records[rebuilt.row[i]] for i in rebuilt.row)
largest = max(np.abs(synced.vectors[synced.row[i]] - rebuilt.vectors[rebuilt.row[i]]).max() for i in rebuilt.row)
same_top = all([r.id for r in synced.search(embedder.embed_query(p.question))] ==
               [r.id for r in rebuilt.search(embedder.embed_query(p.question))] for p in DAY03_PROBES)
print(f"synced vs rebuilt from scratch: same chunk IDs {same_ids} | same records {same_records} | "
      f"largest vector difference {largest:.1e} | same top {TOP_K} for Day 03's 13 questions {same_top}\n")

# An ingest that failed and returned nothing must not wipe the index
try:
    sync(load_index(INDEX_PATH, index.model, index.revision), [], embedder.embed)
except RuntimeError as error:
    guard_message = str(error)
    print(f"empty chunk list: {guard_message}")

synced vs rebuilt from scratch: same chunk IDs True | same records True | largest vector difference 0.0e+00 | same top 3 for Day 03's 13 questions True

empty chunk list: refusing to delete 44 of 44 chunks. Is the new chunk list incomplete (a failed or partial ingest)? Raise max_delete_fraction if this is intended.


The synced index is the index you'd get by rebuilding from scratch, for a fraction of the embedding work. That equivalence is the property to test in any incremental pipeline, because incremental code is where stale data hides. And the delete guard turns "the ingest returned nothing" into an error instead of an empty search index.

**Deleting from the index is not deleting the data.** The old `index.npz` may live on in backups, an embedding cache (like Day 04's, keyed on chunk hash) may still hold the vector, and logs may contain retrieved text. A real deletion request (for example under data-protection law) has to reach all of them. Day 14 comes back to retention.

## 6. A metadata filter is not access control

Section 3 used `allowed_groups` as a filter, and it worked: an employee's search didn't return the managers-only chunk. It's tempting to call that access control. It isn't, yet. A filter only says *which rows to return for these filter values*. Whether a user is protected depends on everything around it. Four ways it fails:

In [16]:
PAY_QUESTION = "When do pay rises take effect?"
q = embedder.embed_query(PAY_QUESTION)


def groups_filter(groups) -> dict:
    return {"allowed_groups": {"any": sorted(groups)}}


access_hits = {}
print(PAY_QUESTION)
for label, where in [("employee, filter applied", groups_filter({"all-employees"})),
                     ("manager, filter applied", groups_filter({"all-employees", "managers"})),
                     ("1. the filter is forgotten (where=None)", None),
                     ("2. an employee's request says groups = managers", groups_filter({"all-employees", "managers"}))]:
    access_hits[label] = index.search(q, where=where)
    print(f"  {label}")
    show(access_hits[label])

When do pay rises take effect?
  employee, filter applied
    #1 0.59  [expense-reimbursement-policy#approval-and-payment]
    #2 0.58  [parental-leave-policy#returning-to-work]
    #3 0.57  [expense-reimbursement-policy#submitting-a-claim]
  manager, filter applied
    #1 0.70  [manager-pay-review-guide#overview]   <- only managers, people-team
    #2 0.59  [expense-reimbursement-policy#approval-and-payment]
    #3 0.58  [parental-leave-policy#returning-to-work]
  1. the filter is forgotten (where=None)
    #1 0.70  [manager-pay-review-guide#overview]   <- only managers, people-team
    #2 0.59  [expense-reimbursement-policy#approval-and-payment]
    #3 0.58  [parental-leave-policy#returning-to-work]
  2. an employee's request says groups = managers
    #1 0.70  [manager-pay-review-guide#overview]   <- only managers, people-team
    #2 0.59  [expense-reimbursement-policy#approval-and-payment]
    #3 0.58  [parental-leave-policy#returning-to-work]


1. **The filter is optional.** It is a parameter, and any code path that leaves it out (a debug endpoint, a new feature, an evaluation script, a "similar documents" widget) returns the restricted chunk at #1.
2. **The filter trusts its input.** The index can't tell a manager from an employee whose request *says* "managers": both calls are identical. If the groups come from the request, a URL parameter or anything the client controls, anyone can claim any group.

The third failure is about time. The index holds a **copy** of each document's permissions, taken at the last sync. When the system of record changes them, the index doesn't know:

In [17]:
def restrict_travel_policy(sample: Path) -> None:
    # In the system of record, the travel policy becomes managers-only
    acl_path = sample / "access-control.json"
    acl = json.loads(acl_path.read_text(encoding="utf-8"))
    acl["permissions"]["acme-handbook/travel-policy.pdf"] = ["managers"]
    acl_path.write_text(json.dumps(acl, indent=2), encoding="utf-8", newline="\n")


TRAVEL_QUESTION = "What is the hotel limit per night in Singapore?"
q = embedder.embed_query(TRAVEL_QUESTION)
employee = groups_filter({"all-employees"})
live = load_index(INDEX_PATH, index.model, index.revision)     # the index as it was saved before the change

print(f"{TRAVEL_QUESTION}   [employee, filter applied]")
print("  the permission changed, the index hasn't been synced yet")
stale_hits = live.search(q, where=employee)
show(stale_hits)

embedded.clear()
acl_plan = sync(live, corpus_after(restrict_travel_policy), counting_embed)
acl_embedded = sum(embedded)
print(f"  sync: {len(acl_plan['update metadata'])} records updated, {acl_embedded} texts embedded")
synced_hits = live.search(q, where=employee)
show(synced_hits)

What is the hotel limit per night in Singapore?   [employee, filter applied]
  the permission changed, the index hasn't been synced yet
    #1 0.71  [travel-policy#daily-limits-by-city]
    #2 0.53  [expense-reimbursement-policy#meals]
    #3 0.53  [travel-policy#booking-travel]
  sync: 5 records updated, 0 texts embedded
    #1 0.53  [expense-reimbursement-policy#meals]
    #2 0.49  [expense-reimbursement-policy#travel]
    #3 0.48  [remote-work-policy#equipment-and-home-office]


3. **The filter checks yesterday's permissions.** Until the sync ran, the employee still got the travel policy's hotel limits as the top result, although the system of record had restricted the policy to managers. The sync itself was cheap: 5 records updated, nothing re-embedded, because no text changed. How long that window stays open is a design decision. A nightly sync means up to a day of exposure after every revocation; permission changes deserve their own, more frequent sync, or a check against the system of record when results are returned.

In [18]:
with np.load(INDEX_PATH, allow_pickle=False) as data:     # no index code, no filter: just read the file
    raw_records = json.loads(data["records"].tobytes().decode("utf-8"))
    raw_vectors = data["vectors"]
print(f"{INDEX_PATH.name}: {len(raw_records)} chunk texts and {len(raw_vectors)} vectors\n")
restricted_raw = [r for r in raw_records if "all-employees" not in r["allowed_groups"]]
for r in restricted_raw:
    print(f"[{r['chunk_id']}] only {', '.join(r['allowed_groups'])}:")
    print(f"  {r['text'].splitlines()[3]}")

index.npz: 44 chunk texts and 44 vectors

[manager-pay-review-guide#overview] only managers, people-team:
  Changes are approved by the department head and the People Team, and take effect from 1 April. Send questions to people-team@acme.example.


4. **The index file holds everything.** No filter runs when someone reads the file itself: every chunk's text, including the managers-only one, sits in `index.npz`, next to vectors that can be partly turned back into text (Day 04, section 15). The index needs the same protection as the documents: file permissions, encryption at rest, and access to the server it runs on.

So what turns a filter into access control? The filter itself stays; what changes is who controls its input and whether it can be skipped. A minimal sketch:

In [19]:
# Stands in for your identity provider (Entra ID, Okta, LDAP, ...): who is in which group, read on every request
DIRECTORY = {"alice": {"all-employees"}, "maria": {"all-employees", "managers"}}


def retrieve_for(user_id: str, question: str, k: int = TOP_K) -> list[Result]:
    # The only search function the application exposes. `user_id` comes from the verified login, never from
    # the request body.
    #   - the groups come from the directory, not from the caller
    #   - the permission filter is always applied: there is no parameter to turn it off
    #   - an unknown user has no groups and gets nothing; a chunk without groups matches nobody (fail closed)
    groups = DIRECTORY.get(user_id, set())
    if not groups:
        return []
    return index.search(embedder.embed_query(question), k, where=groups_filter(groups))


enforced_hits = {}
for user in ("alice", "maria", "mallory"):
    enforced_hits[user] = retrieve_for(user, PAY_QUESTION)
    print(f"{user}: {PAY_QUESTION}")
    show(enforced_hits[user])

DIRECTORY["maria"] = {"all-employees"}       # maria moves to a role without reports: one change in the directory
print(f"\nmaria, after leaving the managers group (no index change): {PAY_QUESTION}")
enforced_hits["maria, later"] = retrieve_for("maria", PAY_QUESTION)
show(enforced_hits["maria, later"])

alice: When do pay rises take effect?
    #1 0.59  [expense-reimbursement-policy#approval-and-payment]
    #2 0.58  [parental-leave-policy#returning-to-work]
    #3 0.57  [expense-reimbursement-policy#submitting-a-claim]
maria: When do pay rises take effect?
    #1 0.70  [manager-pay-review-guide#overview]   <- only managers, people-team
    #2 0.59  [expense-reimbursement-policy#approval-and-payment]
    #3 0.58  [parental-leave-policy#returning-to-work]
mallory: When do pay rises take effect?
    (no results)

maria, after leaving the managers group (no index change): When do pay rises take effect?
    #1 0.59  [expense-reimbursement-policy#approval-and-payment]
    #2 0.58  [parental-leave-policy#returning-to-work]
    #3 0.57  [expense-reimbursement-policy#submitting-a-claim]


Now the groups come from a source the user doesn't control, the filter can't be skipped, and an unknown user gets nothing instead of everything. A group change in the directory takes effect on the next question, with no index update. A permission change on a *document* still needs a sync (failure 3), so permission syncs should run often, and they are cheap: no re-embedding.

This is still a sketch. A real system also verifies who the user is (authentication), keeps tenants apart, logs who retrieved what, re-checks permissions on the final results before they reach a prompt, and tests all of it with hostile inputs. That is **Day 14**. Today's point is narrower: **a filter is a tool for access control, not access control itself.**

## 7. Do we need a vector database?

Everything above runs on a NumPy matrix. Vector databases (and libraries such as FAISS, usearch, hnswlib) add **approximate nearest-neighbour (ANN)** indexes, for example HNSW graphs, which avoid scoring every vector and return *most* of the true top k much faster at large scale. The cost: an extra dependency, an index to build and tune, and results that are no longer guaranteed exact.

Is it worth it here? Exact search does the same work whatever the vectors contain, so random vectors measure its cost at sizes our corpus doesn't reach:

In [20]:
def median_ms(fn, repeats: int = 15) -> float:
    times = []
    for _ in range(repeats):
        started = time.perf_counter()
        fn()
        times.append((time.perf_counter() - started) * 1000)
    return statistics.median(times)


questions = [p.question for p in DAY03_PROBES]
embed_ms = statistics.median(median_ms(lambda: embedder.embed_query(question), repeats=3) for question in questions)
query = embedder.embed_query(QUESTION)

rng = np.random.default_rng(5)     # random unit vectors: exact search costs the same whatever the vectors contain
print(f"{'chunks':>9}{'vectors in memory':>20}{'exact top 3':>14}")
for n in (len(index), 1_000, 10_000, 100_000):
    vectors = index.vectors if n == len(index) else rng.standard_normal((n, 384), dtype=np.float32)
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    ms = median_ms(lambda: top_k(vectors @ query, TOP_K))
    print(f"{n:>9,}{vectors.nbytes / 1e6:>17.1f} MB{ms:>11.3f} ms")
print(f"\nembedding one question with the model: {embed_ms:.1f} ms (median of {len(questions)} questions)")
print(f"a million chunks would need {1_000_000 * 384 * 4 / 1e9:.1f} GB of vectors in memory")

   chunks   vectors in memory   exact top 3
       44              0.1 MB      0.007 ms
    1,000              1.5 MB      0.031 ms
   10,000             15.4 MB      0.378 ms


  100,000            153.6 MB      3.878 ms

embedding one question with the model: 5.8 ms (median of 13 questions)
a million chunks would need 1.5 GB of vectors in memory


Exact search grows linearly with the number of vectors. On the machine that produced the saved outputs, 100,000 vectors took a few milliseconds per question, less than the model needed to embed the question itself, and 154 MB of memory. Our 44 chunks take microseconds. A million chunks would mean 1.5 GB of vectors and, at the same rate, tens of milliseconds per question (an extrapolation, not measured here). That is roughly where an approximate index starts to earn its place.

**Decision for this series: no vector library yet.** At this size a library would add a dependency, an index to build and approximate results, and save no measurable time. LEARN.md (section 9) compares local options with their licences and measured install sizes, for when your collection grows. What doesn't change with a library: the records, the filters, the sync logic and the access-control questions above. Those are the parts of this notebook that carry over to any vector database.

## 8. Sanity checks

Each check pins a result quoted in this notebook or in LEARN.md. Similarity scores are compared with a tolerance (CPUs differ in the last decimal places); IDs, counts and ranks exactly.

In [21]:
def close(x: float, expected: float, tol: float = 0.01) -> bool:
    return abs(x - expected) <= tol


# Input and the index (sections 1 and 2)
assert len(documents) == 11 and len(chunks) == len(index) == 44 and index.vectors.shape == (44, 384)
assert index.revision == "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
dads = index.search(embedder.embed_query(QUESTION))
assert [r.id for r in dads] == ["parental-leave-policy#returning-to-work", "parental-leave-policy#who-is-eligible",
                                "parental-leave-policy#leave-entitlement"]          # Day 04's result
assert close(dads[0].score, 0.72) and close(dads[2].score, 0.66)


# Metadata filters (section 3)
assert [sum(matches(r, where) for r in index.records) for where in FILTERS.values()] == [5, 39, 43, 44]
assert [doc for doc, _ in hidden] == ["it-software-requests", "manager-pay-review-guide"]
admin = embedder.embed_query(ADMIN_QUESTION)
assert index.search(admin)[0].id == "it-software-requests#approval-times"
assert len(index.search(admin, where=fresh)) == 3
assert not any(r.doc_id == "it-software-requests" for r in (h.chunk for h in index.search(admin, where=fresh)))

# Pre- vs post-filter (section 3.1)
pay, vpn = (embedder.embed_query(question) for question, _ in CASES)
employee_only, one_document = FILTERS["permissions: an employee"], FILTERS["scope: one document"]
assert index.search(pay)[0].id == "manager-pay-review-guide#overview" and close(index.search(pay)[0].score, 0.70)
assert len(index.search(pay, where=employee_only, filter_mode="post")) == 2
assert len(index.search(pay, where=employee_only)) == 3
assert all(r.id.startswith("vpn-access-guide#") for r in index.search(vpn))
assert index.search(vpn, where=one_document, filter_mode="post") == []
assert index.search(vpn, where=one_document)[0].id == "remote-work-policy#security-while-remote"

# Persistence (section 4)
assert same and crash_outcome == {"naive save": "EOFError", "atomic save": 44}
assert not list(OUT_DIR.glob("*.tmp")) and not list(OUT_DIR.glob("crash-demo-*"))
assert "rebuild the index" in refusal

# Sync (section 5)
assert {action: len(ids) for action, ids in scenario_plan.items()} == {
    "add": 2, "re-embed": 1, "update metadata": 11, "delete": 5, "unchanged": 27}
assert scenario_plan["add"] == ["annual-leave-policy#entitlement", "security-incident-reporting#mistakes-to-avoid"]
assert scenario_plan["re-embed"] == ["remote-work-policy#equipment-and-home-office"]
assert scenario_plan["delete"] == ["security-incident-reporting#what-not-to-do", "vpn-access-guide#connecting",
                                   "vpn-access-guide#installing-the-client", "vpn-access-guide#troubleshooting",
                                   "vpn-access-guide#when-to-use-the-vpn"]
assert dict(changed_fields) == {"allowed_groups": 4, "doc_hash": 7} and scenario_embedded == 3
assert len(synced) == len(new_chunks) == 41 and len(upsert_only) == 46
vpn_q, incident_q = "How do I connect to the VPN?", "What should I avoid doing after a security incident?"
assert all(r.id.startswith("vpn-access-guide#") for r in lifecycle_hits[vpn_q, "upsert only"])
assert lifecycle_hits[vpn_q, "full sync"][0].id == "remote-work-policy#security-while-remote"
assert not any(r.id.startswith("vpn-access-guide#") for r in lifecycle_hits[vpn_q, "full sync"])
assert {"security-incident-reporting#what-not-to-do", "security-incident-reporting#mistakes-to-avoid"} <= \
       {r.id for r in lifecycle_hits[incident_q, "upsert only"]}
assert "security-incident-reporting#what-not-to-do" not in {r.id for r in lifecycle_hits[incident_q, "full sync"]}
assert same_ids and same_records and largest < 1e-5 and same_top
assert guard_message.startswith("refusing to delete 44 of 44 chunks")

# A filter is not access control (section 6)
restricted = "manager-pay-review-guide#overview"
assert restricted not in [r.id for r in access_hits["employee, filter applied"]]
assert all(hits[0].id == restricted for label, hits in access_hits.items() if label != "employee, filter applied")
assert stale_hits[0].id == "travel-policy#daily-limits-by-city"
assert len(acl_plan["update metadata"]) == 5 and acl_embedded == 0
assert not any(r.chunk.doc_id == "travel-policy" for r in synced_hits)
travel = embedder.embed_query(TRAVEL_QUESTION)
assert live.search(travel, where=groups_filter({"managers"}))[0].id == "travel-policy#daily-limits-by-city"
assert [r["chunk_id"] for r in restricted_raw] == [restricted] and "1 April" in restricted_raw[0]["text"]
assert restricted not in [r.id for r in enforced_hits["alice"]] and enforced_hits["maria"][0].id == restricted
assert enforced_hits["mallory"] == [] and restricted not in [r.id for r in enforced_hits["maria, later"]]

# The saved index was never modified by the demos: it still holds Day 03's 44 chunks
final = load_index(INDEX_PATH, embedder.model["repo"], embedder.revision)
assert final.records == index.records and np.array_equal(final.vectors, index.vectors)


print("All Day 05 sanity checks passed.")

All Day 05 sanity checks passed.


## 9. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing settings.

1. **Over-fetching.** Add a `fetch` argument to `search()`: in post-filter mode, rank the top `fetch` results, filter, then keep k. How large must `fetch` be before the VPN question scoped to the remote work policy returns 3 results? Why can no fixed number guarantee k results for every filter?
2. **A rename is a delete plus an add.** Rename `vpn-access-guide.md` to `vpn-guide.md` in a `corpus_after()` change and sync. Every chunk gets a new `chunk_id` (the file name is the document ID), so all of them are deleted and re-embedded, although no text changed. Add a lookup by `chunk_hash` to `sync()` so that identical text reuses its vector. What could go wrong if two documents contain an identical chunk?
3. **A permission-only sync.** Write `sync_permissions(index, acl_path)` that reads `access-control.json` and updates `allowed_groups` on every record without re-ingesting or re-chunking anything. How often could you afford to run it compared with a full sync? Which of section 6's failures does it shrink?
4. **One more filter operator.** Add `{"in": [...]}` for single-value fields (for example `doc_id` in a list of documents) and use it to search two documents at once. Then search with a misspelled field name: what happens, and why is that better than returning no results?
5. **Try a library.** Rebuild the index in a local vector database of your choice: insert, update, delete by `doc_id`, filtered search. Check its documentation for its filter mode (pre or post?) and how deletes are applied. Which of section 6's four failures does it solve on its own?

## Summary

- **A vector index is vectors + records + the model.** Keyed by `chunk_id`, an exact NumPy index searched 44 chunks in microseconds and 100,000 random vectors in a few milliseconds. Add an approximate index when measurements say so, not before.
- **Filters are hard rules, and where they run matters.** Pre-filtering fills the top k from the matching records; post-filtering returned 2 of 3 results for a permission filter and 0 of 3 for a document scope. A freshness filter hid the only chunk that answered a question.
- **The lifecycle is part of retrieval.** Save atomically and refuse a mismatched model. Sync by `chunk_id` and `chunk_hash`: 3 of 41 texts re-embedded, with the same result as a rebuild. Never skip deletes (the retired VPN guide kept answering), and refuse a sync that would delete most of the index.
- **A filter is not access control** until its input comes from verified identity, it can't be skipped, the permissions it checks are fresh, and the stored index is protected like the documents.

**Next — Day 06: the first end-to-end grounded RAG.** The index retrieves; next, a model answers from what it retrieved, with citations to `chunk_id`s, and says "I don't know" when the evidence isn't there.